# Sentiment Analysis on Restaurant Reviews

This project adapts the trainer's Restaurant Reviews notebook. It keeps the review-cleaning and Bag-of-Words workflow, fits the vocabulary on training data only, and reports evaluation metrics.

## Dataset

Place Restaurant_Reviews.tsv in this folder. The notebook expects two columns named Review and Liked. The trainer archive did not include the dataset. If NLTK reports that the English stop-word list is missing, download it once with nltk.download('stopwords').

In [ ]:
from pathlib import Path
import re

import nltk
import pandas as pd
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import MultinomialNB


## Load and check the review data

In [ ]:
data_path = Path("Restaurant_Reviews.tsv")
if not data_path.exists():
    raise FileNotFoundError(
        "Place Restaurant_Reviews.tsv next to this notebook before running it."
    )

dataset = pd.read_csv(data_path, sep="\t", quoting=3)
required_columns = {"Review", "Liked"}
missing_columns = required_columns.difference(dataset.columns)
if missing_columns:
    raise ValueError(f"Missing required columns: {sorted(missing_columns)}")

dataset = dataset.dropna(subset=["Review", "Liked"]).copy()
dataset["Liked"] = dataset["Liked"].astype(int)
dataset.head()


## Clean the review text

The trainer example uses regular-expression cleaning, lowercasing, stop-word removal, and Porter stemming. This notebook keeps those steps and prepares the stop-word set once.

In [ ]:
stop_words = set(stopwords.words("english"))
stemmer = PorterStemmer()

def clean_review(text):
    words = re.sub(r"[^a-zA-Z]", " ", str(text)).lower().split()
    return " ".join(
        stemmer.stem(word)
        for word in words
        if word not in stop_words
    )

cleaned_reviews = dataset["Review"].map(clean_review)
labels = dataset["Liked"].to_numpy()


## Split, vectorize, and train

CountVectorizer turns reviews into word-count features. Fit it only on the training reviews so test vocabulary does not leak into training.

In [ ]:
X_train_text, X_test_text, y_train, y_test = train_test_split(
    cleaned_reviews,
    labels,
    test_size=0.20,
    random_state=0,
    stratify=labels,
)

vectorizer = CountVectorizer(max_features=1500)
X_train = vectorizer.fit_transform(X_train_text)
X_test = vectorizer.transform(X_test_text)

classifier = MultinomialNB()
classifier.fit(X_train, y_train)


## Evaluate the model

In [ ]:
y_pred = classifier.predict(X_test)

print("Accuracy:", round(accuracy_score(y_test, y_pred), 3))
print("Confusion matrix (rows=true, columns=predicted; labels 0, 1):")
print(confusion_matrix(y_test, y_pred, labels=[0, 1]))
print(classification_report(
    y_test,
    y_pred,
    labels=[0, 1],
    target_names=["Not liked", "Liked"],
    zero_division=0,
))


## Predict one new review

In [ ]:
new_review = "The food was delicious and the service was friendly."
new_review_clean = clean_review(new_review)
new_review_vector = vectorizer.transform([new_review_clean])
prediction = classifier.predict(new_review_vector)[0]
print("Positive" if prediction == 1 else "Negative")
